In [2]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

Python executable:
C:\Users\DELL\anaconda3\envs\qafza-mlops\python.exe

Python version:
3.12.14 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:37:13) [MSC v.1942 64 bit (AMD64)]


# Experiment 2 – Modeling and Evaluation

This experiment evaluates whether the improved leakage-safe feature set provides better predictive performance than Experiment 1.

The validation set is used for model comparison and model selection. The test set is kept untouched until the final evaluation.

## 1. Load Experiment 2 Data

In [3]:
import numpy as np
from scipy.sparse import load_npz

# Load Experiment 2 processed feature matrices
X_train = load_npz("../artifacts/X_train_experiment2.npz")
X_validation = load_npz("../artifacts/X_validation_experiment2.npz")
X_test = load_npz("../artifacts/X_test_experiment2.npz")

# Load target labels
y_train = np.load(
    "../artifacts/y_train_experiment2.npy",
    allow_pickle=True
)

y_validation = np.load(
    "../artifacts/y_validation_experiment2.npy",
    allow_pickle=True
)

y_test = np.load(
    "../artifacts/y_test_experiment2.npy",
    allow_pickle=True
)

print("Train:", X_train.shape)
print("Validation:", X_validation.shape)
print("Test:", X_test.shape)

print("\nTarget shapes:")
print("y_train:", y_train.shape)
print("y_validation:", y_validation.shape)
print("y_test:", y_test.shape)

print("\nTarget classes:")
print("Train:", np.unique(y_train))
print("Validation:", np.unique(y_validation))
print("Test:", np.unique(y_test))

Train: (67533, 84)
Validation: (14471, 84)
Test: (14472, 84)

Target shapes:
y_train: (67533,)
y_validation: (14471,)
y_test: (14472,)

Target classes:
Train: ['Late' 'On Time']
Validation: ['Late' 'On Time']
Test: ['Late' 'On Time']


## 2. Baseline Model

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score, classification_report

# Baseline model
baseline = DummyClassifier(strategy="most_frequent")

# Train only on training data
baseline.fit(X_train, y_train)

# Evaluate on validation data
y_validation_pred_baseline = baseline.predict(X_validation)

baseline_balanced_accuracy = balanced_accuracy_score(
    y_validation,
    y_validation_pred_baseline
)

print("Baseline Balanced Accuracy:", baseline_balanced_accuracy)

print("\nBaseline Classification Report:")
print(
    classification_report(
        y_validation,
        y_validation_pred_baseline,
        target_names=["Late", "On Time"],
        zero_division=0
    )
)

Baseline Balanced Accuracy: 0.5

Baseline Classification Report:
              precision    recall  f1-score   support

        Late       0.00      0.00      0.00       773
     On Time       0.95      1.00      0.97     13698

    accuracy                           0.95     14471
   macro avg       0.47      0.50      0.49     14471
weighted avg       0.90      0.95      0.92     14471



## 3. Logistic Regression

In [5]:
from sklearn.linear_model import LogisticRegression

# Logistic Regression model
logistic_model = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    solver="liblinear",
    max_iter=2000,
    random_state=42
)

# Train only on the training data
logistic_model.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


## 4. Logistic Regression Evaluation

In [6]:
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# Predict on validation set
y_validation_pred_lr = logistic_model.predict(X_validation)

# Balanced Accuracy
lr_balanced_accuracy = balanced_accuracy_score(
    y_validation,
    y_validation_pred_lr
)

print("Logistic Regression Balanced Accuracy:", lr_balanced_accuracy)

print("\nClassification Report:")
print(
    classification_report(
        y_validation,
        y_validation_pred_lr,
        target_names=["Late", "On Time"],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_validation,
        y_validation_pred_lr
    )
)

Logistic Regression Balanced Accuracy: 0.6294814664967474

Classification Report:
              precision    recall  f1-score   support

        Late       0.18      0.35      0.24       773
     On Time       0.96      0.91      0.93     13698

    accuracy                           0.88     14471
   macro avg       0.57      0.63      0.58     14471
weighted avg       0.92      0.88      0.90     14471


Confusion Matrix:
[[  271   502]
 [ 1255 12443]]


## 5. Threshold Optimization

In [8]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Get probability scores for the Late class
y_validation_proba_lr = logistic_model.predict_proba(X_validation)

# Find the probability column corresponding to Late
late_class_index = list(logistic_model.classes_).index("Late")

late_probabilities = y_validation_proba_lr[:, late_class_index]

# Thresholds to evaluate
thresholds = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20]

threshold_results = []

for threshold in thresholds:

    y_pred_threshold = np.where(
        late_probabilities >= threshold,
        "Late",
        "On Time"
    )

    threshold_results.append({
        "Threshold": threshold,
        "Balanced Accuracy": balanced_accuracy_score(
            y_validation,
            y_pred_threshold
        ),
        "Late Precision": precision_score(
            y_validation,
            y_pred_threshold,
            pos_label="Late",
            zero_division=0
        ),
        "Late Recall": recall_score(
            y_validation,
            y_pred_threshold,
            pos_label="Late",
            zero_division=0
        ),
        "Late F1": f1_score(
            y_validation,
            y_pred_threshold,
            pos_label="Late",
            zero_division=0
        )
    })

threshold_results = pd.DataFrame(threshold_results)

print(threshold_results)

   Threshold  Balanced Accuracy  Late Precision  Late Recall   Late F1
0       0.50           0.629481        0.177588     0.350582  0.235755
1       0.45           0.643834        0.152471     0.419146  0.223602
2       0.40           0.659242        0.133747     0.501940  0.211214
3       0.35           0.670715        0.116314     0.597671  0.194731
4       0.30           0.666596        0.098448     0.689521  0.172297
5       0.25           0.655525        0.084400     0.802070  0.152728
6       0.20           0.637340        0.074803     0.909444  0.138236


## 6. Select Final Model

Based on the validation results, Logistic Regression with the improved feature set is selected as the final model for Experiment 2.

Threshold optimization showed that a threshold of **0.35** achieved the highest validation Balanced Accuracy (0.671) while also providing a substantially higher Recall for the Late class (0.598) compared with the default threshold of 0.50.

Therefore, a threshold of **0.35** is selected as the operating threshold for the final evaluation of Experiment 2.

In [9]:
# Select the final model and threshold for Experiment 2

final_model = logistic_model
final_threshold = 0.35

print("Experiment 2 final model: Logistic Regression")
print("Experiment 2 final threshold:", final_threshold)

Experiment 2 final model: Logistic Regression
Experiment 2 final threshold: 0.35


## 7. Final Test Evaluation

The selected Logistic Regression model with the improved feature set is evaluated once on the previously untouched test set using the selected threshold of 0.35.

The test set is used only to provide an unbiased estimate of the final model's generalization performance.

In [10]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# Predict probabilities for the Late class
y_test_proba = final_model.predict_proba(X_test)

# Find the probability column corresponding to Late
late_class_index = list(final_model.classes_).index("Late")

late_probabilities_test = y_test_proba[:, late_class_index]

# Apply the selected threshold
y_test_pred = np.where(
    late_probabilities_test >= final_threshold,
    "Late",
    "On Time"
)

# Final evaluation
test_accuracy = accuracy_score(y_test, y_test_pred)
test_balanced_accuracy = balanced_accuracy_score(y_test, y_test_pred)

print("Experiment 2 Test Accuracy:", test_accuracy)
print("Experiment 2 Test Balanced Accuracy:", test_balanced_accuracy)

print("\nExperiment 2 Test Classification Report:")
print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=["Late", "On Time"],
        zero_division=0
    )
)

print("\nExperiment 2 Test Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_test_pred
    )
)

Experiment 2 Test Accuracy: 0.6870508568269762
Experiment 2 Test Balanced Accuracy: 0.612042001398655

Experiment 2 Test Classification Report:
              precision    recall  f1-score   support

        Late       0.11      0.53      0.18       957
     On Time       0.95      0.70      0.81     13515

    accuracy                           0.69     14472
   macro avg       0.53      0.61      0.49     14472
weighted avg       0.90      0.69      0.77     14472


Experiment 2 Test Confusion Matrix:
[[ 503  454]
 [4075 9440]]


## 8. Experiment 2 Summary

Experiment 2 introduced additional leakage-safe temporal and geographical features, including estimated delivery duration, approval delay, calendar-based features, and geographical distance.

Using the improved feature set, Logistic Regression with a threshold of 0.35 achieved substantially better performance on the untouched test set than Experiment 1.

The final test results were 68.7% Accuracy, 61.2% Balanced Accuracy, 52.6% Recall for the Late class, 11.0% Precision, and 18.3% F1-score.

These results indicate that the additional prediction-time features improved the model's ability to identify delayed orders and improved its generalization performance.

## 9. Save Final Experiment 2 Model and Results

The final Logistic Regression model from Experiment 2 and its selected classification threshold are saved as project artifacts. The final test performance is also stored to support reproducibility and comparison with Experiment 1.

In [11]:
import joblib
import json

# Save the final Experiment 2 model
joblib.dump(
    final_model,
    "../artifacts/final_logistic_model_experiment2.joblib"
)

# Save the selected threshold
with open(
    "../artifacts/final_threshold_experiment2.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {"threshold": float(final_threshold)},
        f,
        indent=4
    )

# Save final test results
final_results_experiment2 = {
    "model": "Logistic Regression",
    "threshold": float(final_threshold),
    "test_accuracy": float(test_accuracy),
    "test_balanced_accuracy": float(test_balanced_accuracy),
    "late_precision": 0.11,
    "late_recall": 0.53,
    "late_f1": 0.18
}

with open(
    "../artifacts/final_results_experiment2.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_results_experiment2,
        f,
        indent=4
    )

print("Experiment 2 final model and results saved successfully.")

Experiment 2 final model and results saved successfully.


## 10. Experiment 1 vs Experiment 2

In [12]:
import pandas as pd

# Final comparison of Experiment 1 and Experiment 2

final_comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Balanced Accuracy",
        "Late Precision",
        "Late Recall",
        "Late F1"
    ],
    "Experiment 1": [
        0.6519,
        0.4651,
        0.0500,
        0.2500,
        0.0920
    ],
    "Experiment 2": [
        0.6871,
        0.6120,
        0.1100,
        0.5266,
        0.1830
    ]
})

print(final_comparison)

              Metric  Experiment 1  Experiment 2
0           Accuracy        0.6519        0.6871
1  Balanced Accuracy        0.4651        0.6120
2     Late Precision        0.0500        0.1100
3        Late Recall        0.2500        0.5266
4            Late F1        0.0920        0.1830


### Final Comparison Interpretation

Experiment 2 outperformed Experiment 1 across all reported test metrics.

The most notable improvement was observed in the detection of late orders. Late-class Recall increased from 25.0% in Experiment 1 to 52.6% in Experiment 2, while Balanced Accuracy increased from 46.5% to 61.2%.

Late-class Precision also improved from 5.0% to 11.0%, and the Late-class F1-score increased from 9.2% to 18.3%.

These results indicate that the additional leakage-safe temporal and geographical features improved the model's ability to generalize to previously unseen orders.

## 11. Final Conclusion

Experiment 2 was selected as the preferred final approach because it achieved better performance than Experiment 1 across all evaluated test metrics.

The improved leakage-safe feature set, including geographical and temporal information, increased Balanced Accuracy from 46.5% to 61.2% and improved Late-class Recall from 25.0% to 52.7%.

Although the model still produces a relatively high number of false positive late-delivery alerts, the improvement demonstrates that the additional prediction-time features provide useful information for identifying potentially delayed orders.

The final selected model is Logistic Regression with a classification threshold of 0.35.